In [0]:
CATALOG = "de_prac"
SCHEMA = "day01"
VOLUME = "data-files"

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
INCOMING_PATH = f"{BASE_PATH}/incoming"

CUSTOMER_COUNT = 100_000
ORDER_COUNT = 500_000

print("Incoming path:", INCOMING_PATH)
print("Customers:", CUSTOMER_COUNT)
print("Orders:", ORDER_COUNT)

In [0]:
from pyspark.sql.functions import col, concat, lit, when, expr

customers_df = (
    spark.range(1, CUSTOMER_COUNT + 1)
    .withColumnRenamed("id", "customer_id")
    .withColumn(
        "customer_name",
        concat(lit("Customer_"), col("customer_id"))
    )
    .withColumn(
        "email",
        when(
            col("customer_id") % 100 == 0,
            lit(None)
        ).otherwise(
            concat(lit("customer"), col("customer_id"), lit("@example.com"))
        )
    )
    .withColumn(
        "state",
        expr("""
            element_at(
                array('OH','TX','CA','NY','NJ'),
                CAST((customer_id % 5) + 1 AS INT)
            )
        """)
    )
)

customers_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"{INCOMING_PATH}/customers")

print("Customer records created:", customers_df.count())

display(customers_df.limit(10))

In [0]:
from pyspark.sql.functions import col, when, expr, round

orders_df = (
    spark.range(1, ORDER_COUNT + 1)
    .withColumnRenamed("id", "order_id")
    .withColumn(
        "customer_id",
        when(
            col("order_id") % 250 == 0,
            lit(999999)   # intentionally invalid customer
        ).otherwise(
            (col("order_id") % CUSTOMER_COUNT) + 1
        )
    )
    .withColumn(
        "order_amount",
        when(
            col("order_id") % 500 == 0,
            lit(None)
        ).otherwise(
            round((col("order_id") % 500) + 10.50, 2)
        )
    )
    .withColumn(
        "status",
        expr("""
            element_at(
                array('NEW','SHIPPED','DELIVERED','CANCELLED'),
                CAST((order_id % 4) + 1 AS INT)
            )
        """)
    )
    .withColumn(
        "order_date",
        expr("date_sub(current_date(), CAST(order_id % 30 AS INT))")
    )
)

orders_df.write \
    .mode("overwrite") \
    .json(f"{INCOMING_PATH}/orders")

print("Order records created:", orders_df.count())

display(orders_df.limit(10))

In [0]:
customers_check = (
    spark.read
    .option("header", "true")
    .csv(f"{INCOMING_PATH}/customers")
)

orders_check = (
    spark.read
    .json(f"{INCOMING_PATH}/orders")
)

print("Customers:", customers_check.count())
print("Orders:", orders_check.count())